# Day 1 — 데이터 스키마 확정 + 무결성/누수 감사

**담당: 김진영. CPU only. 정확한 분석이 속도보다 우선한다 — 30분은 목표일 뿐 넘어도 된다.**
그동안 나머지 팀원은 라벨 검수(300~500개)를 진행하므로 대기 시간이 낭비되지 않는다.

이 노트북의 1번 목적은 다른 4명이 기다리는 **스키마 확정**이다. 2절(스키마 자동 추론)이
핵심 산출물이고, 그 결과물(`data_meta/schema_report.md`와 `configs/base.yaml`에 붙일
YAML 조각)을 6절에서 팀 채팅에 그대로 붙일 수 있는 형태로 다시 출력한다.

**GPU를 전혀 쓰지 않는다.** `torch`/`transformers`를 이 노트북에서 import하지 않는다 —
GPU가 필요한 작업(B, C)은 이 노트북이 끝난 뒤에 시작한다.

각 절은 끝나는 대로 디스크에 저장한다. 중간에 막혀도 앞 절 결과는 남는다.

## 1. 압축 해제 및 인벤토리

암호는 셀에서 직접 입력받는다 — **절대 하드코딩하지 않는다.**

In [ ]:
import getpass
import zipfile
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
print("REPO_ROOT:", REPO_ROOT)

ZIP_PATH = REPO_ROOT / "ssafy-16-2-ai.zip"
DATA_DIR = REPO_ROOT / "data"
DATA_META_DIR = REPO_ROOT / "data_meta"
DATA_META_DIR.mkdir(exist_ok=True)

if not ZIP_PATH.exists():
    raise SystemExit(f"{ZIP_PATH} 가 없습니다. 배포 원본을 저장소 루트에 먼저 받아두세요.")

if DATA_DIR.exists() and any(DATA_DIR.iterdir()):
    print(f"{DATA_DIR} 에 이미 파일이 있습니다 — 압축 해제를 건너뜁니다.")
else:
    password = getpass.getpass("ZIP 암호를 입력하세요 (화면에 표시되지 않습니다): ")
    DATA_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(DATA_DIR, pwd=password.encode("utf-8"))
    print(f"압축 해제 완료: {DATA_DIR}")
    del password  # 변수에도 남겨두지 않는다

### 실제 파일 수를 09/17 metadata 분석값과 대조

기준값(`tools/zip_stats.py` 09/17 실측): 총 16,118 / JPG 16,111 / CSV 4 / ipynb 3,
Train 6,714 / Dev 2,683 / Test 6,714. **다르면 배포본이 바뀐 것이므로 크게 경고한다.**

In [ ]:
EXPECTED = {
    "total": 16118, "jpg": 16111, "csv": 4, "ipynb": 3,
    "train": 6714, "dev": 2683, "test": 6714,
}

all_files = [p for p in DATA_DIR.rglob("*") if p.is_file()]
actual = {
    "total": len(all_files),
    "jpg": sum(1 for p in all_files if p.suffix.lower() == ".jpg"),
    "csv": sum(1 for p in all_files if p.suffix.lower() == ".csv"),
    "ipynb": sum(1 for p in all_files if p.suffix.lower() == ".ipynb"),
    "train": sum(1 for p in all_files if "train" in p.parts),
    "dev": sum(1 for p in all_files if "dev" in p.parts),
    "test": sum(1 for p in all_files if "test" in p.parts),
}

print(f"{'항목':8} {'기대값':>10} {'실측값':>10} {'일치':>6}")
mismatch = False
for k in EXPECTED:
    ok = EXPECTED[k] == actual[k]
    mismatch = mismatch or not ok
    print(f"{k:8} {EXPECTED[k]:>10} {actual[k]:>10} {'OK' if ok else '!!'}")

if mismatch:
    print("\n" + "*" * 60)
    print("*** 경고: 파일 수가 09/17 기준과 다릅니다 — 배포본이 바뀌었을 수 있습니다! ***")
    print("*** 팀에 반드시 알리고 계속 진행할지 판단하세요.                    ***")
    print("*" * 60)
else:
    print("\n일치 확인됨 — 09/17 배포본과 동일합니다.")

In [ ]:
baseline_notebook_paths = sorted(DATA_DIR.rglob("*.ipynb"))
print(f"발견된 베이스라인 노트북: {len(baseline_notebook_paths)}개")
for p in baseline_notebook_paths:
    print(f"  {p.name}  ({p.stat().st_size:,} bytes)")

if not baseline_notebook_paths:
    print("\n*** 경고: ipynb를 못 찾았습니다 — 09/17 기준 3개가 있어야 합니다. ***")

## 2. 스키마 자동 추론 ★ 핵심 산출물

CSV 4개 각각: 행 수 / 컬럼명 / dtype / 결측치, 컬럼별 고유값 수와 샘플 5개.
정답 컬럼 형식(a/b/c/d? 0~3? 텍스트?), dev.csv에 정답이 있는지(없으면 validation으로
못 쓴다), 이미지 1장당 문항 수(1이 아니면 group split 기준이 반드시 이미지여야 한다 —
같은 이미지의 문항이 train/val로 갈리면 누수)를 확인한다.

In [ ]:
# --- 정답 컬럼 형식 특정 ---
from src.data import _CANDIDATES  # noqa: E402 — 후보 컬럼명 재사용

def guess_answer_column(df: pd.DataFrame) -> str | None:
    for cand in _CANDIDATES["answer"]:
        for col in df.columns:
            if cand in col.lower():
                return col
    return None

answer_format_report = {}
for name, df in dfs.items():
    if name == "extra":
        continue
    ans_col = guess_answer_column(df)
    if ans_col is None:
        answer_format_report[name] = "정답 컬럼 없음 (test는 정상, dev/train이면 문제)"
        continue
    values = df[ans_col].dropna().unique()
    sample = sorted(str(v) for v in values)[:10]
    answer_format_report[name] = f"컬럼={ans_col}, 고유값 예시={sample}"

print("정답 컬럼 형식:")
for name, report in answer_format_report.items():
    print(f"  [{name}] {report}")

if "dev" in answer_format_report and "없음" in answer_format_report["dev"]:
    print("\n*** 경고: dev.csv에 정답이 없습니다 — 로컬 validation Accuracy를 못 잽니다! ***")

In [ ]:
# --- 이미지 1장당 문항 수 ---
from src.data import _CANDIDATES as _C

def guess_image_column(df: pd.DataFrame) -> str | None:
    for cand in _C["image"]:
        for col in df.columns:
            if cand in col.lower():
                return col
    return None

questions_per_image = {}
for name, df in dfs.items():
    if name == "extra":
        continue
    img_col = guess_image_column(df)
    if img_col is None:
        print(f"[{name}] 이미지 컬럼을 못 찾음")
        continue
    n_rows = len(df)
    n_images = df[img_col].nunique()
    ratio = n_rows / n_images if n_images else float("nan")
    questions_per_image[name] = ratio
    print(f"[{name}] 행={n_rows} 고유이미지={n_images} 문항/이미지={ratio:.2f}")

if any(r != 1.0 for r in questions_per_image.values()):
    print("\n이미지당 문항 수가 1이 아닙니다 — group split 기준은 반드시 이미지여야 합니다")
    print("(같은 이미지의 여러 문항이 train/val로 갈리면 그 자체가 누수입니다).")

### `configs/base.yaml`에 붙일 YAML 조각 + 리포트 저장

In [ ]:
import yaml

primary_df = dfs.get("train", next(iter(dfs.values())))
id_col = None
for cand in _C["id"]:
    for col in primary_df.columns:
        if cand in col.lower():
            id_col = col
            break
    if id_col:
        break

question_col = None
for cand in _C["question"]:
    for col in primary_df.columns:
        if cand in col.lower():
            question_col = col
            break
    if question_col:
        break

choices_col = None
for cand in _C["choices"]:
    for col in primary_df.columns:
        if cand in col.lower():
            choices_col = col
            break
    if choices_col:
        break

img_col_final = guess_image_column(primary_df)
ans_col_final = guess_answer_column(primary_df)

columns_yaml = {
    "data": {
        "columns": {
            "id": id_col,
            "image": img_col_final,
            "question": question_col,
            "choices": choices_col,
            "choice_prefixes": ["A", "B", "C", "D"],
            "answer": ans_col_final,
        }
    }
}

yaml_fragment = yaml.dump(columns_yaml, allow_unicode=True, sort_keys=False)
print("--- configs/base.yaml 의 data.columns 에 붙여넣을 조각 ---")
print(yaml_fragment)

In [ ]:
schema_report_lines = [
    "# 스키마 확정 리포트 (Day 1, 김진영)",
    "",
    "## 파일 인벤토리",
    "",
]
for k in EXPECTED:
    schema_report_lines.append(f"- {k}: 기대={EXPECTED[k]} 실측={actual[k]} {'OK' if EXPECTED[k]==actual[k] else '!! 불일치'}")

schema_report_lines += ["", "## 정답 컬럼 형식", ""]
for name, report in answer_format_report.items():
    schema_report_lines.append(f"- {name}: {report}")

schema_report_lines += ["", "## 이미지당 문항 수", ""]
for name, ratio in questions_per_image.items():
    schema_report_lines.append(f"- {name}: {ratio:.2f}")

schema_report_lines += ["", "## configs/base.yaml data.columns 조각", "", "```yaml", yaml_fragment.rstrip(), "```"]

schema_report_path = DATA_META_DIR / "schema_report.md"
schema_report_path.write_text("\n".join(schema_report_lines), encoding="utf-8")
print(f"저장됨: {schema_report_path}")

## 3. 무결성 · 누수

09/17에 (file_size, CRC32)로 잡은 중복 후보를 **내용 SHA-256으로 재검증**한다
(CRC32는 충돌 가능하므로 여기서 확정한다). split 간 중복도 재확인한다.
**Train↔Test 중복이 실재하면 그 자체가 중요한 정보다.**

In [ ]:
import hashlib
from collections import defaultdict

def sha256_of(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

all_images = [p for p in DATA_DIR.rglob("*.jpg")]
print(f"이미지 {len(all_images)}장 SHA-256 계산 중... (수 분 걸릴 수 있음)")

hash_to_paths: dict[str, list[Path]] = defaultdict(list)
for i, p in enumerate(all_images):
    hash_to_paths[sha256_of(p)].append(p)
    if (i + 1) % 2000 == 0:
        print(f"  {i+1}/{len(all_images)}")

dup_groups = {h: paths for h, paths in hash_to_paths.items() if len(paths) > 1}
print(f"\nSHA-256 기준 중복 그룹: {len(dup_groups)}개, 총 {sum(len(v) for v in dup_groups.values())}장")

In [ ]:
def split_of(path: Path) -> str | None:
    for part in path.parts:
        if part in ("train", "dev", "test"):
            return part
    return None

cross_split_pairs = defaultdict(int)
for h, paths in dup_groups.items():
    splits_in_group = {split_of(p) for p in paths}
    if len(splits_in_group) > 1:
        splits_sorted = tuple(sorted(s for s in splits_in_group if s))
        for i in range(len(splits_sorted)):
            for j in range(i + 1, len(splits_sorted)):
                cross_split_pairs[(splits_sorted[i], splits_sorted[j])] += 1

print("split 간 중복 쌍(SHA-256 기준):")
for pair, count in cross_split_pairs.items():
    print(f"  {pair[0]}<->{pair[1]}: {count}쌍")
    if "train" in pair and "test" in pair:
        print("  *** Train<->Test 중복 확인됨 — 이건 크게 보고해야 합니다 (리키지/제출 전략에 영향) ***")

In [ ]:
# --- src/data.py의 group_split + check_leakage ---
from src.config import load_config
from src.data import load_split, group_split, check_leakage

try:
    cfg = load_config("9b_base_1024")  # data.columns가 비어 있으면 자동 추론 경로를 탄다
    train_samples, schema = load_split(cfg, "train")
    tr, val = group_split(train_samples, val_fraction=cfg.get("data.split.val_fraction", 0.1),
                           seed=cfg.seed, cache=cfg.path("data.split.cache") / "train_val.json")
    leakage = check_leakage(tr, val)
    print("group_split + check_leakage 결과:")
    print(leakage)
    if not leakage["clean"]:
        print("*** 경고: group split에 누수가 있습니다! ***")
except Exception as exc:
    print("group_split/check_leakage 실행 실패(스키마가 아직 안 맞을 수 있음):", exc)

## 4. 이미지 분포

크기(px) min/p50/p90/max, split별. 종횡비 분포. max 지점 이미지 몇 장을 실제로 열어
확인한다(꼬리에서 OOM이 난다). 이 분포로 `input.long_side` 상한을 정할 수 있는지 본다.

In [ ]:
from PIL import Image, ImageFilter
from collections import Counter
import statistics

ORIENTATION_TAG = 274  # EXIF Orientation. 1=정상, 2~8=회전/미러. 태그가 없으면 getexif()에 키 자체가 없다.

def pct(sorted_vals, p):
    if not sorted_vals:
        return 0
    k = (len(sorted_vals) - 1) * p
    f, c = int(k), min(int(k) + 1, len(sorted_vals) - 1)
    return sorted_vals[f] if f == c else sorted_vals[f] + (sorted_vals[c] - sorted_vals[f]) * (k - f)

def edge_variance(gray_small) -> float:
    # 값싼 블러 추정치 — 에지 응답의 분산. 낮을수록 흐릿함(판독 불가 의심).
    edges = gray_small.filter(ImageFilter.FIND_EDGES)
    pixels = list(edges.getdata())
    return statistics.pvariance(pixels) if pixels else 0.0

# ★ 전수 이미지를 한 번만 연다. 크기/종횡비/EXIF회전/색상모드/밝기/블러/손상을 이 루프에서 다 잰다.
#   (예전에는 이 수집을 4절과 4-1절에서 두 번 돌려 16,111장을 중복 디코딩했다 — 한 번으로 합쳤다.)
size_by_split = defaultdict(list)
aspect_by_split = defaultdict(list)
orientation_by_split = defaultdict(Counter)
mode_by_split = defaultdict(Counter)
brightness_by_split = defaultdict(list)
blur_by_split = defaultdict(list)
max_size_paths = []
corrupt_images = []

for i, p in enumerate(all_images):
    s = split_of(p) or "other"
    try:
        with Image.open(p) as im:
            exif = im.getexif()
            # 태그가 없으면 None. '태그 없음(None)'과 '정상(1)'을 반드시 구분한다 —
            # None을 'Orientation != 1'로 세면 EXIF 없는 데이터셋이 전부 '회전 이상'으로 잘못 잡힌다.
            orientation = exif.get(ORIENTATION_TAG)  # 없으면 None
            orientation_by_split[s][orientation] += 1

            w, h = im.size
            if orientation in (5, 6, 7, 8):  # 90도 회전 계열 — 보정 후 가로/세로가 뒤바뀐다
                w, h = h, w
            long_side = max(w, h)
            size_by_split[s].append(long_side)
            aspect_by_split[s].append(w / h if h else 0)
            max_size_paths.append((long_side, p))
            mode_by_split[s][im.mode] += 1

            gray_small = im.convert("L").resize((128, 128))
            brightness_by_split[s].append(statistics.mean(gray_small.getdata()))
            blur_by_split[s].append(edge_variance(gray_small))
    except Exception as exc:
        corrupt_images.append((p, str(exc)))
    if (i + 1) % 2000 == 0:
        print(f"  {i+1}/{len(all_images)} 처리됨")

print("=== 크기(px) / 종횡비 ===")
for s, sizes in size_by_split.items():
    sv = sorted(sizes)
    print(f"[{s}] n={len(sv)} min={sv[0]} p50={pct(sv,0.5):.0f} p90={pct(sv,0.9):.0f} max={sv[-1]}")
    av = sorted(aspect_by_split[s])
    print(f"     종횡비 min={av[0]:.2f} p50={pct(av,0.5):.2f} max={av[-1]:.2f}")


### 3-1. Perceptual hash — SHA-256이 못 잡는 근접 중복

CLAUDE.md의 "Day 1 운영 원칙"은 "SHA-256뿐 아니라 perceptual hash로도 한다 —
resize/crop된 유사 이미지까지"라고 정해뒀는데, 위 3절은 **바이트가 완전히 같은
경우(exact duplicate)만** 잡는다. 같은 사진을 다른 크기로 내보냈거나 재인코딩·살짝
crop한 버전은 SHA-256으로는 안 걸린다. `imagehash`가 필요하다(`pip install imagehash`).

**시간이 걸려도 끝까지 돌린다.** 정확한 분석이 30분 목표보다 우선이고, 그동안 나머지
팀원은 라벨 검수를 진행한다. 쌍 비교는 numpy로 벡터화해 전수(약 1.3억 쌍)를 본다.

In [ ]:
try:
    import imagehash
    import numpy as np

    # phash 64비트. 임계값 6은 너무 느슨해 서로 다른 장면이 사슬로 엮인다
    # (2026-09-21 실측: dHash/phash 거리 4~8 후보 대부분이 픽셀상 다른 장면이었다).
    # 2로 낮추고, 연결 성분(그룹)으로 묶지 않고 쌍 단위로만 보고한다.
    HASH_DISTANCE_THRESHOLD = 2

    phash_to_paths: dict = defaultdict(list)
    for i, p in enumerate(all_images):
        try:
            with Image.open(p) as im:
                phash_to_paths[imagehash.phash(im)].append(p)
        except Exception:
            continue
        if (i + 1) % 2000 == 0:
            print(f"  phash {i+1}/{len(all_images)}")

    def _stats(path):
        # 사람이 '같은 장면인가'를 눈으로 보기 전에 숫자로 거르게 한다.
        with Image.open(path) as im:
            g = np.asarray(im.convert("L").resize((32, 32)), dtype=np.float32)
        return path.stat().st_size, float(g.mean()), float(g.std()), g

    # 1) phash가 완전히 같고 split이 다른 쌍 (거리 0)
    pairs = []
    for group in phash_to_paths.values():
        if len(group) > 1:
            for a in range(len(group)):
                for b in range(a + 1, len(group)):
                    if split_of(group[a]) != split_of(group[b]):
                        pairs.append((group[a], group[b], 0))

    # 2) phash 거리 1~2 인 쌍 (쌍 단위. 연결 성분으로 묶지 않는다)
    hashes = list(phash_to_paths.keys())
    hash_ints = np.array([int(str(h), 16) for h in hashes], dtype=np.uint64)
    for i in range(len(hashes) - 1):
        xor = np.bitwise_xor(hash_ints[i + 1:], hash_ints[i])
        dist = np.unpackbits(xor.view(np.uint8).reshape(-1, 8), axis=1).sum(axis=1)
        for k in np.nonzero((dist >= 1) & (dist <= HASH_DISTANCE_THRESHOLD))[0]:
            j = i + 1 + int(k)
            pi, pj = phash_to_paths[hashes[i]][0], phash_to_paths[hashes[j]][0]
            if split_of(pi) != split_of(pj):
                pairs.append((pi, pj, int(dist[k])))

    print(f"split 간 근접 중복 후보(phash 거리 <= {HASH_DISTANCE_THRESHOLD}): {len(pairs)}쌍")
    if pairs:
        print(f"{'pair':44} {'dist':>4} {'MAE':>6}  size_a/size_b   bright_a/b   std_a/b   판정")
        for a, b, d in pairs:
            sa, ma, sda, ga = _stats(a)
            sb, mb, sdb, gb = _stats(b)
            mae = float(np.abs(ga - gb).mean())
            verdict = "동일/재인코딩 의심" if mae <= 4 else "다른 장면(오탐 의심)"
            print(f"  {a.name} ~ {b.name:18} {d:>4} {mae:>6.1f}  {sa:>7}/{sb:<7} "
                  f"{ma:>5.0f}/{mb:<5.0f} {sda:>5.0f}/{sdb:<5.0f}  {verdict}")
        print("\n※ phash 거리만으로 '중복'이라 단정하지 않는다. 32x32 gray 픽셀 MAE<=4면 동일/재인코딩,")
        print("  그 이상이면 서로 다른 장면일 가능성이 높다 — 위 수치로 사람이 최종 확인한다.")
    else:
        print("split 간 phash 거리 <= 2 근접 중복 없음.")
except ImportError as exc:
    print(f"{exc.name} 미설치 — 근접 중복 검사를 건너뜀 (pip install imagehash numpy).")
    print("CLAUDE.md의 perceptual hash 원칙은 이 라이브러리가 있어야 실행된다.")

In [ ]:
max_size_paths.sort(reverse=True)
print("최대 크기 이미지 top 5 (실제로 열어서 확인):")
for size, p in max_size_paths[:5]:
    print(f"  {size}px  {p}")
    try:
        with Image.open(p) as im:
            im.load()  # 실제로 디코딩해서 손상 여부까지 확인
            print(f"    -> 정상 로드됨, size={im.size}, mode={im.mode}")
    except Exception as exc:
        print(f"    -> 로드 실패: {exc}")

## 5. 질문 template 분석 ★ Day 1 최우선

숫자·고유명사를 마스킹한 뒤 빈도 상위 패턴을 뽑는다. 유형 태그는
**`configs/base.yaml`의 `evaluate.types`에 이미 고정된 taxonomy**
(Overall/OCR/Number/TimeDate/Table/Logo/Scene/Object/Relation/Composite)를 그대로 쓴다 —
새 분류 체계를 만들지 않는다.

In [ ]:
# 손상(로드 실패) 이미지 요약 — 위 4절 셀의 전수 1회 순회에서 이미 수집했다.
# 여기서 이미지를 다시 열지 않는다(중복 디코딩 제거). corrupt_images / *_by_split 를 그대로 쓴다.
print(f"=== 로드 실패(손상 의심) 이미지: {len(corrupt_images)}장 (전수 검사) ===")
for p, err in corrupt_images[:20]:
    print(f"  {p}: {err}")
if len(corrupt_images) > 20:
    print(f"  ... 외 {len(corrupt_images) - 20}장 더")


In [ ]:
if question_col and question_col in primary_df.columns:
    tags_df = pd.DataFrame({"question": primary_df[question_col], "tag": tags})
    question_tags_path = DATA_META_DIR / "question_tags.csv"
    # BOM: 사람이 Excel로 여는 산출물. cp949 Windows에서도 한글이 안 깨진다.
    tags_df.to_csv(question_tags_path, index=False, encoding="utf-8-sig")
    print(f"저장됨: {question_tags_path}")

## 4-1. 이미지 변형 요인 종합 점검 — 회전·색상모드·밝기·블러

증강(augmentation) 여부를 판단하기 전에 "실제로 그런 이미지가 있는가"부터 값싸게 확인한다.
전부 CPU/PIL만으로 **4절 전수 1회 순회**에 얹어서 쟀다 — 추가 다운로드·모델·재디코딩 불필요.

- **EXIF Orientation** — 원래 휴대폰 사진은 픽셀을 안 돌리고 "화면에 낼 때 몇 도 돌려라"는
  Orientation 태그만 남기는 경우가 흔하다. 그런데 **이 대회 데이터셋은 EXIF가 제거돼 있다**
  (주최측이 짧은 변 720px로 강제 리사이즈하며 메타데이터를 벗긴 것으로 보인다). 즉
  Orientation 분포는 사실상 `{None: 전부}`다. **EXIF가 없으므로 실제 회전 여부는 태그로
  판정할 수 없다 — 회전 augmentation을 논하려면 픽셀을 직접 봐야 한다.** 태그 없음(None)을
  "Orientation != 1"로 세면 전량이 '회전 이상'으로 잘못 집계되므로, 아래 집계는
  None(태그 없음) / 1(정상) / 2\~8(실제 회전) 세 갈래로 나눈다.
- **색상 모드(RGB/L/CMYK/RGBA 등)** — RGB가 아닌 이미지가 섞여 있으면 전처리에서
  조용히 깨지거나 색이 달라질 수 있다.
- **밝기** — CLAUDE.md가 이미 난이도 요인으로 꼽은 "조명·그림자·저조도"를 숫자로 확인한다.
- **블러(에지 분산)** — 낮으면 초점이 안 맞거나 흔들린 사진일 가능성 — 판독 자체가
  불가능한 이미지일 수 있다(Accuracy 상한과 직결, 라벨 감사 강제 포함 대상 후보).

In [ ]:
print("=== EXIF Orientation 분포 (None=EXIF 없음, 1=정상, 2~8=실제 회전) ===")
n_no_exif = 0   # 태그 자체가 없음 (getexif에 Orientation 키 부재)
n_normal = 0    # Orientation == 1
n_rotated = 0   # 2~8: 실제로 회전/미러가 걸린 이미지
for s, counter in orientation_by_split.items():
    print(f"[{s}]", dict(counter))
    for k, v in counter.items():
        if k is None:
            n_no_exif += v
        elif k == 1:
            n_normal += v
        else:
            n_rotated += v
if n_rotated:
    print(f"  *** 실제 회전(Orientation 2~8) {n_rotated}장 — 픽셀 방향 보정 필요 ***")

print("\n=== 색상 모드 분포 ===")
non_rgb_total = 0
for s, counter in mode_by_split.items():
    print(f"[{s}]", dict(counter))
    non_rgb_total += sum(v for k, v in counter.items() if k != "RGB")

print("\n=== 밝기(그레이스케일 평균, 0~255) — 낮을수록 저조도 ===")
for s, vals in brightness_by_split.items():
    sv = sorted(vals)
    print(f"[{s}] min={sv[0]:.1f} p10={pct(sv,0.1):.1f} p50={pct(sv,0.5):.1f} p90={pct(sv,0.9):.1f} max={sv[-1]:.1f}")

print("\n=== 블러 추정치(edge variance) — 낮을수록 흐릿함 ===")
for s, vals in blur_by_split.items():
    sv = sorted(vals)
    print(f"[{s}] min={sv[0]:.1f} p10={pct(sv,0.1):.1f} p50={pct(sv,0.5):.1f}")

print("\n" + "=" * 70)
if n_rotated:
    print(f"EXIF Orientation 2~8 이미지 {n_rotated}장 — 실제 회전이 있다.")
    print("src/data.py build_views()의 ImageOps.exif_transpose()가 이들을 올바른 방향으로 보정한다.")
elif n_no_exif and not n_normal:
    print(f"EXIF 없음 {n_no_exif}장 — Orientation 태그가 전부 부재. exif_transpose는 no-op다.")
    print("이 데이터셋은 회전 여부를 태그로 판정할 수 없다 — 픽셀을 직접 봐야 한다.")
else:
    print(f"EXIF Orientation 요약: 없음 {n_no_exif} / 정상 {n_normal} / 회전 {n_rotated}.")
if non_rgb_total:
    print(f"*** {non_rgb_total}장이 RGB가 아님 — build_views()가 .convert('RGB')를 적용하므로 전처리는 안전하나 원인 확인 권장.")
print("=" * 70)


In [ ]:
print("=" * 60)
print("Day 1 데이터 스키마 확정 요약 — 김진영")
print("=" * 60)
print(f"\n[파일 인벤토리] 09/17 기준과 {'일치' if not mismatch else '불일치!!'}")
for k in EXPECTED:
    print(f"  {k}: {actual[k]}")

print("\n[정답 컬럼 형식]")
for name, report in answer_format_report.items():
    print(f"  {name}: {report}")

print("\n[이미지당 문항 수]")
for name, ratio in questions_per_image.items():
    print(f"  {name}: {ratio:.2f}")

print(f"\n[Train<->Test 중복] {cross_split_pairs.get(('test','train'), cross_split_pairs.get(('train','test'), 0))}쌍")

print(f"\n[이미지 손상] 로드 실패 {len(corrupt_images)}장")

# EXIF Orientation: None(태그 없음) / 1(정상) / 2~8(실제 회전) 세 갈래. None을 회전으로 세지 않는다.
_orient = Counter()
for counter in orientation_by_split.values():
    _orient.update(counter)
_n_no_exif = _orient.get(None, 0)
_n_rotated = sum(v for k, v in _orient.items() if k not in (None, 1))
if _n_rotated:
    print(f"[EXIF 회전] 실제 회전(2~8) {_n_rotated}장 — src/data.py exif_transpose가 보정")
elif _n_no_exif:
    print(f"[EXIF 회전] 없음 — EXIF 태그 부재 {_n_no_exif}장. exif_transpose는 no-op")
else:
    print("[EXIF 회전] 실제 회전 0장")
_non_rgb = sum(
    sum(v for k, v in counter.items() if k != "RGB") for counter in mode_by_split.values()
)
print(f"[색상 모드] RGB 아님  {_non_rgb}장")

if question_col and question_col in primary_df.columns:
    print("\n[질문 유형 비율 상위 5]")
    print(tag_ratio.head(5))

print("\n[configs/base.yaml에 붙여넣을 조각]")
print(yaml_fragment)

print("\n스키마 리포트: data_meta/schema_report.md")
print("질문 태그: data_meta/question_tags.csv")
print("=" * 60)
